# E02b: независимое воспроизведение на Google Colab (T4)

Ноутбук исполняет **сценарий A** из
[`reports/2026-10-22-colab-reproduction-plan.md`](../reports/2026-10-22-colab-reproduction-plan.md):
прогнать перепись E02b на портативном lite-пакете (18 МБ) на другом железе (T4)
и другой версии torch, сравнить **классы** с локальной переписью.

Почему Colab: для Kaggle GPU требовалась верификация телефона, которая не
прошла; Colab даёт бесплатный T4 без неё.

**Что нужно заранее (один раз):**

1. Runtime → Change runtime type → **T4 GPU**.
2. Положить на Google Drive файл `e02b-portable.zip` из `dist/kaggle-dataset/`
   в папку `MyDrive/morphogenesis/`.
3. Запустить ячейки по порядку.

**Что этот прогон не делает:** не считает частоту `struct_2` и не обучает — это
перепись тех же весов на другом стеке. Частоты смешивать между железом нельзя.


In [ ]:
# 1. Окружение и Drive
from google.colab import drive
drive.mount('/content/drive')

import json, os, platform, shutil, subprocess, sys, zipfile
from pathlib import Path

import torch

print('python  :', sys.version.split()[0])
print('torch   :', torch.__version__)
print('cuda    :', torch.version.cuda)
print('device  :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
if torch.cuda.is_available():
    print('vram GB :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print('platform:', platform.platform())

DRIVE = Path('/content/drive/MyDrive/morphogenesis')
DRIVE.mkdir(parents=True, exist_ok=True)
print('drive dir:', DRIVE)


In [ ]:
# 2. Распаковать портативный пакет с Drive
ZIP = DRIVE / 'e02b-portable.zip'
assert ZIP.is_file(), f'положи {ZIP} на Drive (файл из dist/kaggle-dataset/e02b-portable.zip)'

PKG = Path('/content/e02b-portable')
if PKG.exists():
    shutil.rmtree(PKG)
PKG.mkdir(parents=True)
with zipfile.ZipFile(ZIP) as zf:
    zf.extractall(PKG)

manifest = json.loads((PKG / 'portable_manifest.json').read_text(encoding='utf-8'))
print('manifest :', manifest['schema_version'], manifest['kind'])
print('source   :', manifest['source']['git_commit_short'], 'dirty=', manifest['source']['git_dirty'])
print('families :', len(manifest['package']['families']), 'checkpoints:', manifest['package']['n_checkpoints'])
print('files    :', manifest['package']['n_files'], 'bytes:', manifest['package']['total_bytes'])
print('census   :', manifest['datasets']['census']['sha256'][:16], '(packaged)')


In [ ]:
# 3. Перепись на GPU T4 (smoke: 3 семейства x 16 стартов x 2000 шагов)
# Полный режим: --mode full --seeds 0-63 (дольше, ~40-60 мин на T4)
CMD = [sys.executable, 'verify_portable.py', '--mode', 'smoke', '--device', 'cuda',
       '--out', 'verify_report.colab.json']
print(' '.join(CMD))
proc = subprocess.run(CMD, cwd=PKG, text=True, capture_output=True)
print(proc.stdout[-4000:])
if proc.returncode != 0:
    print('STDERR:', proc.stderr[-4000:])
raise SystemExit(proc.returncode) if proc.returncode else None


In [ ]:
# 4. Итог и сохранение отчёта на Drive
report_path = PKG / 'verify_report.colab.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
summary = report.get('summary', {})
print('status   :', report['status'])
print('mode     :', report['mode'], 'device:', report['device'])
print('runs     :', summary.get('n_runs'), 'mismatches:', summary.get('n_mismatches'))
print('max diff :', summary.get('max_abs_descriptor_diff'))
print('census   :', (report.get('census') or {}).get('sha256', '')[:16])

out = DRIVE / 'verify_report.colab.json'
shutil.copy2(report_path, out)
print('saved to :', out)
print('\nСкопируй отчёт в репозиторий: dist/kaggle-report/colab/verify_report.json')


## Сценарий B (полные чекпоинты) — по решению владельца

Без полноценной тренировки это не «независимое воспроизведение обучения», а
проверка переноса **карты „сид → класс“** между железом: те же полные веса,
другая карта. Нужны 3 чекпоинта по 540 МБ на Drive:
`E02c_R-9_s2000_seed42`, `WP4_R-9_s2000_seed101`, `WP4_R-9_s2000_seed142`
(+ untrained-контроль создаётся кодом).

E02c-плечо идёт основной серией (`--families`), известные сиды — как reference
(`--include-families`, `arm=REF`): тот же код переписи и та же замороженная
проекция E02b, но в вердикт H-R1/R2/R3 они не входят. Канонического контроля
`WP4_R-9_s2000_seed42` на Drive нет, поэтому перепись идёт с
`--allow-missing-control`: в сводке будет `positive_control_present: false`, а
не подставной контроль.

Ожидания (локальные, для сверки): seed42 → `struct_2` 64/64; seed101 →
`struct_1`; seed142 → `struct_1`. Совпадение классов — цель; дескрипторы могут
дрейфовать, это записывается, а не подгоняется.


In [ ]:
# 5. Сценарий B (опционально): перепись известных сидов на чужом железе
RUN_FULL = False   # поставь True, когда чекпоинты лежат на Drive
CKPT_DRIVE = DRIVE / 'checkpoints'

MAIN = ['E02c_R-9_s2000_seed42']                          # --families (arm=R-9)
REF = ['WP4_R-9_s2000_seed101', 'WP4_R-9_s2000_seed142']  # --include-families (arm=REF)

if RUN_FULL:
    assert CKPT_DRIVE.is_dir(), f'положи чекпоинты в {CKPT_DRIVE}'
    REPO = Path('/content/morphogenesis')
    if not REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/mak777uh/-', str(REPO)], check=True)
    morph = REPO / 'morphogenesis'
    for name in MAIN + REF:
        src = CKPT_DRIVE / name
        assert (src / 'nca_final.pt').is_file(), f'нет {src / "nca_final.pt"}'
        shutil.copytree(src, morph / 'checkpoints' / name, dirs_exist_ok=True)

    OUT = DRIVE / 'census_colab_known_seeds.json'
    SUMMARY_OUT = DRIVE / 'summary_colab_known_seeds.json'
    cmd = [sys.executable, 'e02c_census.py',
           '--families', ','.join(MAIN),
           '--include-families', ','.join(REF),
           '--allow-missing-control',  # контроля WP4 seed42 на Drive нет
           '--seeds', '0-63', '--steps', '2000', '--device', 'cuda',
           '--out', str(OUT), '--summary-out', str(SUMMARY_OUT)]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=morph, check=True)

    summary = json.loads(SUMMARY_OUT.read_text(encoding='utf-8'))
    print('verdict  :', summary.get('verdict'), '| applicable:', summary.get('verdict_applicable'))
    print('control  :', summary.get('positive_control_present'))
    stats = dict(summary.get('stats') or {})
    stats.update(summary.get('reference_stats') or {})
    for fam, st in stats.items():
        print(f'{fam:26s} n={st["n"]:2d} struct_2 {st["struct_2"]:2d} '
              f'stable_frozen {st["struct_2_stable_frozen"]:2d}')
    print('census   :', OUT)
    print('summary  :', SUMMARY_OUT)
else:
    print('сценарий B выключен: RUN_FULL = False')
